In [ ]:
from pathlib import Path
import pynucastro as pyna
import pandas as pd
import random
import matplotlib.pyplot as plt
import numpy as np
#%matplotlib widget

In [ ]:
#original rates
rates_reaclib_18_9_20=pyna.rates.library.Library(libfile=r"Nuclear_Data\run1\Reaclib_18_9_20")

In [ ]:
def reaclib2_to_reaclib1(input_file, output_file):
    """
    Convert a Reaclib v2 format file into Reaclib v1 format.
    
    Parameters
    ----------
    input_file : str or Path
        Path to Reaclib v2 file
    output_file : str or Path
        Path to write Reaclib v1 file
    """
    input_file = Path(input_file)
    output_file = Path(output_file)

    with input_file.open("r") as fin, output_file.open("w") as fout:
        header=[]
        head=0
        for line in fin:
            if len(line) != 75:
                if line[0] != head:
                    head=line[0]
                    header.append(line[0])
                    fout.write(line[0]+' '*73 + '\n')
                    fout.write(' '*74 + '\n')
                    fout.write(' '*74 + '\n')
                else:
                    continue
            else:
                fout.write(line)
   



In [ ]:
#beta decay Q neg
filter_beta=pyna.RateFilter(max_reactants=1,
                            max_products=1,
                            filter_function=lambda r: r.Q<0 and r.reactants[0].Z+1==r.products[0].Z and r.reactants[0].A==r.products[0].A
                            )

#alpha decay Q neg
filter_alpha=pyna.RateFilter(products=['he4'],
                             exact=False,
                             max_reactants=1,
                             max_products=2,
                             filter_function=lambda r: r.Q<0 and r.reactants[0].Z==r.products[1].Z+r.products[0].Z and r.reactants[0].A==r.products[1].A+r.products[0].A
                             )
 
rates_alpha_remove=rates_reaclib_18_9_20.filter(filter_alpha)
rates_beta_remove=rates_reaclib_18_9_20.filter(filter_beta)

for i in rates_alpha_remove.get_rates():
    rates_reaclib_18_9_20.remove_rate(i)
for j in rates_beta_remove.get_rates():
    rates_reaclib_18_9_20.remove_rate(j)

In [ ]:

file=Path(r"Nuclear_Data\run2\Reaclib_Q_positive")
open(file,'w')
rates_reaclib_18_9_20.write_to_file(file)
reaclib2_to_reaclib1(r"Nuclear_Data\run2\Reaclib_Q_positive",
                     r"Nuclear_Data\run2\Reaclib_Q_positive_R1")
